# SNP-level log-likelihood benchmark

Do genomic language models encode, at a SNP position, a nucleotide distribution that tracks the **empirical allele frequency** in the population?

For ~5000 reference-context SNPs per species (Arabidopsis / rice / soy) we read each model's distribution over {A,C,G,T} at the SNP:
- **AgroNT-1B** (6-mer, masked) & **Carbon** (6-mer, causal): mask/read the SNP's 6-mer token, restrict the vocab to the 4 six-mers matching the reference 6-mer except at the SNP offset, renormalize.
- **PlantCaduceus** (single-base, masked): mask the SNP base, read the native 4-way distribution.

Two experiments: **exp1** P(ref base) vs ref-allele freq, **exp2** P(alt base) vs alt-allele freq.

> **Envs:** AgroNT/Carbon run in the `svar` kernel; PlantCaduceus needs `mamba_ssm` so it runs via a subprocess to the `plantcad` env. Both import the same `snpll_lib.py` (written below). Outputs are staged in `OUTDIR`, so cells skip work that already exists — set `FORCE_*` to recompute. Full recompute is ~30–45 min (Arabidopsis VCF stream + PlantCaduceus scoring).

## Config

In [ ]:
import os, sys, subprocess, itertools
import numpy as np, torch

REPO      = "/home/andrew/svar"
NB_DIR    = f"{REPO}/notebooks"
OUTDIR    = "/home/andrew/svar_scratch/snpll"        # prep/score/png outputs (durable scratch)
SVAR_PY   = "/home/andrew/anaconda3/envs/svar/bin/python"
PLANTCAD_PY = "/home/andrew/anaconda3/envs/plantcad/bin/python"
DEVICE    = "cuda:0"
CARBON_SIZE = "500M"          # flip to "3B" for the stronger Carbon (slower)
N_SNPS, SEED = 5000, 42
FORCE_PREP, FORCE_SCORE = False, False   # True = recompute even if outputs exist

SPECIES = ["arabidopsis", "rice", "soy"]
MODELS  = ["agront", "carbon", "plantcad"]     # agront/carbon -> svar; plantcad -> plantcad env
FASTA = {
  "soy":  "/home/andrew/svar_scratch/datasets/soy/Glycine_max.Glycine_max_v2.1.dna_sm.toplevel.fa",
  "rice": "/home/andrew/rice_data/Oryza_sativa.IRGSP-1.0.dna_sm.toplevel.fa",
  "arabidopsis": "/home/andrew/svar_scratch/datasets/arabidopsis/Arabidopsis_thaliana.TAIR10.dna_sm.toplevel.fa"}
VCF = {
  "soy":  "/home/andrew/svar_scratch/datasets/soy/soysnp50k_a2_final.vcf",
  "rice": "/home/andrew/rice_data/sativas413_msu7_final.vcf",
  "arabidopsis": "/home/andrew/svar_scratch/datasets/arabidopsis/arabidopsis_1001g_final.vcf"}

os.makedirs(OUTDIR, exist_ok=True)
os.environ.setdefault("HF_HOME", "/home/andrew/svar_scratch/hf_cache")
for p in (NB_DIR, REPO):
    if p not in sys.path: sys.path.insert(0, p)
def carbon_tag(): return "carbon" if CARBON_SIZE == "500M" else f"carbon{CARBON_SIZE}"
print("OUTDIR:", OUTDIR, "| CARBON_SIZE:", CARBON_SIZE)

## Helper library (`snpll_lib.py`)
Written to disk so both conda envs import the identical logic.

In [ ]:
%%writefile /home/andrew/svar/notebooks/snpll_lib.py
"""snpll_lib.py — prep + scoring for the SNP-level log-likelihood benchmark.

Single source of truth imported by BOTH conda envs:
  * svar     env : prep (pysam/pyfaidx), AgroNT, Carbon
  * plantcad env : PlantCaduceus (needs mamba_ssm)
All heavy imports are lazy (inside the branch that needs them) so importing this
module never pulls mamba into the svar env or transformers-carbon into plantcad.

Method
------
For each SNP we want the model's distribution over {A,C,G,T} at the SNP position,
in reference context (the 5 non-SNP bases of the SNP's 6-mer = reference):
  * k-mer models (agront masked / carbon causal): read the logits for the SNP's
    6-mer token (agront: mask it; carbon: read the preceding position, causal),
    restrict the vocab to the 4 six-mers that equal the reference 6-mer except at
    the SNP offset, and renormalize -> 4-way nucleotide distribution.
  * plantcad (single-base masked): mask the SNP base, read the native 4-way dist.
Empirical target = alt-carrier frequency across the dataset's samples (1 if a
sample carries the alt allele, else 0/missing) — the codebase gt_alts convention.
"""
import numpy as np, torch, torch.nn.functional as F

WIN, HALF = 510, 255          # 510 = 85 clean 6-mers; SNP centered at offset 255
ACGT = set("ACGT")
BASES = "ACGT"


# ============================================================== PREP ==========
def prep_species(species, fasta_path, vcf_path, out, n=5000, seed=42):
    """Reservoir-sample n eligible SNPs, compute alt-carrier freq, extract the
    reference window. Saves a plain-typed dict for the scoring stage."""
    import pysam
    from pyfaidx import Fasta
    rng = np.random.default_rng(seed)
    fa = Fasta(fasta_path)
    cname = {int(k): k for k in fa.keys() if k.isdigit()}
    clen = {c: len(fa[k]) for c, k in cname.items()}
    vf = pysam.VariantFile(vcf_path)
    n_samples = len(vf.header.samples)
    print(f"[{species}] samples={n_samples} chroms={sorted(cname)}", flush=True)

    K = int(n * 1.15)
    res, seen = [], 0
    for rec in vf:
        try:
            chrom = int(rec.chrom)
        except (ValueError, TypeError):
            continue
        if chrom not in cname or not rec.alts or len(rec.alts) != 1:
            continue
        ref, alt = rec.ref, rec.alts[0]
        if ref is None or alt is None or len(ref) != 1 or len(alt) != 1:
            continue
        ref, alt = ref.upper(), alt.upper()
        if ref not in ACGT or alt not in ACGT:
            continue
        pos0 = rec.pos - 1
        if pos0 - HALF < 0 or pos0 + HALF > clen[chrom]:
            continue
        seen += 1
        if len(res) < K:
            keep = len(res)
        else:
            j = int(rng.integers(0, seen)); keep = j if j < K else -1
        if keep < 0:
            continue
        carriers = sum(1 for s in rec.samples.values()
                       if (g := s.get("GT")) is not None and any(x not in (0, None) for x in g))
        entry = {"chrom": chrom, "pos0": pos0, "ref": ref, "alt": alt,
                 "alt_freq": carriers / n_samples}
        if len(res) < K:
            res.append(entry)
        else:
            res[keep] = entry
        if seen % 200000 == 0:
            print(f"  scanned {seen}, reservoir {len(res)}", flush=True)

    ref_seqs, ref_c, alt_c, af, chroms, poss = [], [], [], [], [], []
    mism = 0
    for e in res:
        if len(ref_seqs) >= n:
            break
        a = e["pos0"] - HALF
        seq = str(fa[cname[e["chrom"]]][a:a + WIN]).upper()
        if len(seq) != WIN or "N" in seq:
            continue
        if seq[HALF] != e["ref"]:
            mism += 1; continue
        ref_seqs.append(seq); ref_c.append(e["ref"]); alt_c.append(e["alt"])
        af.append(e["alt_freq"]); chroms.append(e["chrom"]); poss.append(e["pos0"])
    print(f"[{species}] final windows={len(ref_seqs)} ref-mismatch-skipped={mism}", flush=True)
    torch.save({"species": species, "win": WIN, "offset": HALF,
                "ref_seq": ref_seqs, "ref_char": ref_c, "alt_char": alt_c,
                "alt_freq": np.array(af, np.float32), "chrom": chroms, "pos0": poss}, out)
    return out


# ============================================================== SCORING =======
def score_model(model, prep_path, out, device="cuda:0", carbon_size="500M", batch_size=32):
    """Score one model over one prep file; saves p_ref/p_alt/alt_freq."""
    dev = torch.device(device)
    bidx = {b: i for i, b in enumerate(BASES)}
    d = torch.load(prep_path, map_location="cpu", weights_only=False)
    seqs, off = d["ref_seq"], d["offset"]
    ref_c, alt_c, alt_freq = d["ref_char"], d["alt_char"], d["alt_freq"]
    N = len(seqs)
    tk = 1 + off // 6                      # SNP 6-mer token index (cls/<dna> prefix=1)
    k6s = (off // 6) * 6
    o_in = off - k6s
    print(f"[{model}] N={N} tk={tk} o_in={o_in} (carbon_size={carbon_size})", flush=True)

    if model == "agront":
        from transformers import AutoModelForMaskedLM, AutoTokenizer
        name = "InstaDeepAI/agro-nucleotide-transformer-1b"
        tok = AutoTokenizer.from_pretrained(name)
        m = AutoModelForMaskedLM.from_pretrained(name, dtype=torch.bfloat16).to(dev).eval()
        vocab = tok.get_vocab(); kmer_id = lambda s: vocab[s]; mask_id = tok.mask_token_id
    elif model == "carbon":
        from CARBON_modules import load_carbon_lm
        m, tok = load_carbon_lm(repo_id=f"HuggingFaceBio/Carbon-{carbon_size}", device=dev); m.eval()
        kmer_id = lambda s: tok.convert_tokens_to_ids(s)
    else:
        from PlantCAD_modules import load_plantcad_mlm
        m, tok = load_plantcad_mlm(device=dev); m.eval()
        v = tok.get_vocab(); base_ids = torch.tensor([v[b.lower()] for b in BASES]); mask_id = tok.mask_token_id

    @torch.no_grad()
    def batch(bseqs, brefc, baltc):
        if model == "plantcad":
            ids = tok(bseqs, return_tensors="pt", padding=False)["input_ids"].to(dev)
            ids[:, off] = mask_id
            lg = m(input_ids=ids).logits.float()[:, off, :]
            four = lg[:, base_ids.to(dev)]
        else:
            if model == "carbon":
                enc = tok(["<dna>" + s for s in bseqs], return_tensors="pt", add_special_tokens=False)
                ids = enc["input_ids"].to(dev)
                lg = m(input_ids=ids).logits.float()[:, tk - 1, :]     # causal: predict tk from tk-1
            else:
                enc = tok(bseqs, return_tensors="pt"); ids = enc["input_ids"].clone()
                ids[:, tk] = mask_id; ids = ids.to(dev)
                lg = m(input_ids=ids, attention_mask=torch.ones_like(ids)).logits.float()[:, tk, :]
            cand = torch.empty(len(bseqs), 4, dtype=torch.long)
            for r, s in enumerate(bseqs):
                ref6 = s[k6s:k6s + 6]
                for j, b in enumerate(BASES):
                    cand[r, j] = kmer_id(ref6[:o_in] + b + ref6[o_in + 1:])
            four = torch.gather(lg, 1, cand.to(dev))
        p = F.softmax(four, dim=1).cpu().numpy()
        pr = np.array([p[i, bidx[brefc[i]]] for i in range(len(bseqs))])
        pa = np.array([p[i, bidx[baltc[i]]] for i in range(len(bseqs))])
        return pr, pa

    p_ref = np.empty(N, np.float32); p_alt = np.empty(N, np.float32)
    for s in range(0, N, batch_size):
        e = min(N, s + batch_size)
        pr, pa = batch(seqs[s:e], ref_c[s:e], alt_c[s:e]); p_ref[s:e] = pr; p_alt[s:e] = pa
    torch.save({"model": model, "species": d["species"], "carbon_size": carbon_size,
                "p_ref": p_ref, "p_alt": p_alt, "alt_freq": alt_freq,
                "ref_char": ref_c, "alt_char": alt_c}, out)
    r = float(np.corrcoef(p_alt, alt_freq)[0, 1])
    print(f"[{model}/{d['species']}] mean p_ref={p_ref.mean():.3f} p_alt={p_alt.mean():.3f} "
          f"| corr(p_alt,alt_freq)={r:.3f}", flush=True)
    return out


## 1 · Prep — sample SNPs + reference windows (model-agnostic)

In [ ]:
import snpll_lib
# Prep is model-agnostic. Arabidopsis streams a 9.7 GB VCF (~few min); rice/soy quick.
for sp in SPECIES:
    out = f"{OUTDIR}/prep_{sp}.pt"
    if os.path.exists(out) and not FORCE_PREP:
        print("skip (exists):", out); continue
    snpll_lib.prep_species(sp, FASTA[sp], VCF[sp], out, n=N_SNPS, seed=SEED)

## 2 · Score AgroNT + Carbon (svar kernel)

In [ ]:
# AgroNT + Carbon run in THIS (svar) kernel. (Carbon output tag encodes its size.)
import importlib; import snpll_lib; importlib.reload(snpll_lib)
for model, tag in [("agront", "agront"), ("carbon", carbon_tag())]:
    for sp in SPECIES:
        out = f"{OUTDIR}/score_{tag}_{sp}.pt"
        if os.path.exists(out) and not FORCE_SCORE:
            print("skip (exists):", out); continue
        snpll_lib.score_model(model, f"{OUTDIR}/prep_{sp}.pt", out,
                              device=DEVICE, carbon_size=CARBON_SIZE)

## 3 · Score PlantCaduceus (plantcad env via subprocess)

In [ ]:
# PlantCaduceus needs the mamba (plantcad) env -> run via subprocess importing the same lib.
env = dict(os.environ, PYTHONPATH=f"{NB_DIR}:{REPO}")
for sp in SPECIES:
    out = f"{OUTDIR}/score_plantcad_{sp}.pt"
    if os.path.exists(out) and not FORCE_SCORE:
        print("skip (exists):", out); continue
    code = (f"import snpll_lib; snpll_lib.score_model('plantcad',"
            f"r'{OUTDIR}/prep_{sp}.pt', r'{out}', device='{DEVICE}')")
    print("plantcad", sp, "...", flush=True)
    subprocess.run([PLANTCAD_PY, "-c", code], env=env, check=True)

## 4 · Correlation summary

In [ ]:
from scipy.stats import pearsonr, spearmanr
import pandas as pd
CARBON_TAG = carbon_tag()
tags = ["agront", CARBON_TAG, "plantcad"]
rows = []
for m, sp in itertools.product(tags, SPECIES):
    f = f"{OUTDIR}/score_{m}_{sp}.pt"
    if not os.path.exists(f): continue
    d = torch.load(f, map_location="cpu", weights_only=False)
    pref, palt, af = d["p_ref"], d["p_alt"], d["alt_freq"]; reff = 1 - af
    rows.append(dict(model=m, species=sp,
        exp1_r_pref_reffreq=round(pearsonr(reff, pref)[0], 3),
        exp2_r_palt_altfreq=round(pearsonr(af, palt)[0], 3),
        exp2_spearman=round(spearmanr(af, palt)[0], 3),
        mean_pref=round(float(pref.mean()), 3), mean_palt=round(float(palt.mean()), 3)))
df = pd.DataFrame(rows); df

## 5 · Plots
Regenerates `snpll_exp1_refallele.png` and `snpll_exp2_altallele.png`.

In [ ]:
import matplotlib.pyplot as plt
TITLE = {"agront": "AgroNT-1B (6-mer)", "plantcad": "PlantCaduceus (per-base)",
         CARBON_TAG: f"Carbon-{CARBON_SIZE} (6-mer)"}
def calib(x, y, nb=10):
    ed = np.quantile(x, np.linspace(0, 1, nb + 1)); ed[-1] += 1e-9; xs, ys = [], []
    for i in range(nb):
        mm = (x >= ed[i]) & (x < ed[i + 1])
        if mm.sum() >= 5: xs.append(x[mm].mean()); ys.append(y[mm].mean())
    return np.array(xs), np.array(ys)

EXP = {"exp1": (lambda d: 1 - d["alt_freq"], lambda d: d["p_ref"],
                "empirical reference-allele freq", "model P(ref base)", "snpll_exp1_refallele.png"),
       "exp2": (lambda d: d["alt_freq"], lambda d: d["p_alt"],
                "empirical alt-allele freq", "model P(alt base)", "snpll_exp2_altallele.png")}
for exp, (xf, yf, xlab, ylab, fname) in EXP.items():
    fig, axes = plt.subplots(len(SPECIES), len(tags), figsize=(13, 11), sharex=True, sharey=True)
    for i, sp in enumerate(SPECIES):
        for j, m in enumerate(tags):
            ax = axes[i, j]; f = f"{OUTDIR}/score_{m}_{sp}.pt"
            if not os.path.exists(f): ax.set_visible(False); continue
            d = torch.load(f, map_location="cpu", weights_only=False); x, y = xf(d), yf(d)
            ax.hexbin(x, y, gridsize=30, cmap="Blues", mincnt=1, extent=(0, 1, 0, 1))
            cx, cy = calib(x, y); ax.plot(cx, cy, "-o", color="#d1495b", ms=4, lw=1.8)
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
            ax.text(0.04, 0.93, f"r={pearsonr(x, y)[0]:+.3f}", transform=ax.transAxes,
                    va="top", fontsize=10, bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85))
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            if i == 0: ax.set_title(TITLE.get(m, m), fontsize=11)
            if j == 0: ax.set_ylabel(f"{sp}\n{ylab}", fontsize=10)
            if i == len(SPECIES) - 1: ax.set_xlabel(xlab, fontsize=9)
    fig.suptitle(f"{exp}: {ylab} vs {xlab}  (5000 SNPs/species; decile-mean red, y=x dashed)", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.98]); fig.savefig(f"{OUTDIR}/{fname}", dpi=110)
    print("saved", f"{OUTDIR}/{fname}")
    plt.show()